# Install Necessary Libraries

In [ ]:
!pip install open_clip_torch
!pip install torch
!pip install transformers
!pip install pillow
!pip install chromadb
!pip install pandas
!pip install openpyxl
!pip install datasets
!pip install tqdm
!pip install sentence-transformers
!pip install qwen
!pip install qwen-vl-utils
!pip install hf_xet

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


#Install Flash Attention
makes models run faster with less vram usage

note: requires ampere or newer gpu (so T4 will not work with flash attention)

I reccomend using L4 GPU for running this notebook

In [ ]:
!nvidia-smi
!pip install ninja
!pip install flash-attn --no-build-isolation

Tue Jan 27 01:58:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Setup Paths to VDB and Qwen Scripts

In [ ]:
import sys
from google.colab import drive
drive.mount('/content/drive')

# download this folder from my drive to get the scripts and the preprocessed VDB
# https://drive.google.com/drive/folders/1YOmGkaUuylhh3IjivjrnmoOyQxQ9KX3T?usp=sharing
# VDB is currently embedded with Qwen3VL-Embedding-2B

# when you mount your drive, adjust these paths accordingly to your drive
sys.path.append('/content/drive/MyDrive/colab/mcam/')
VDB_PATH = '/content/drive/MyDrive/colab/mcam/VDB'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Embed Keywords into Vector Database
note: this only needs to be run once and then the persistent VDB can be stored so that it doesn't have to be run again.

In [ ]:
import chromadb
import torch
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import os

# Import the hf Qwen wrapper
from scripts.qwen3_vl_embedding import Qwen3VLEmbedder

#Setup ChromaDB
chroma_client = chromadb.PersistentClient(path=VDB_PATH)

try:
    chroma_client.delete_collection(name="aat_terms")
    print("Existing collection 'aat_terms' found and deleted.")
except:
    pass

collection = chroma_client.create_collection(
    name="aat_terms",
    metadata={"hnsw:space": "cosine"}
)

#Load Model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Qwen model on {device}...")

model_name_or_path = "Qwen/Qwen3-VL-Embedding-2B"

# Initialize model with float16 for GPU memory savings if on CUDA
dtype = torch.float16 if device == "cuda" else torch.float32

model = Qwen3VLEmbedder(
    model_name_or_path=model_name_or_path
    # Uncomment below if you have Flash Attention installed for speed
    # torch_dtype=dtype,
    # attn_implementation="flash_attention_2"
)

# --- 3. Load Data ---
print("Loading AAT terms...")
ds = load_dataset("KeeganC/aat-preferred-terms", split="train")#.select(range(128))
print(f"Loaded {len(ds)} terms from AAT.")

# --- 4. Batch Process Text ---
BATCH_SIZE = 24 # Qwen is larger/heavier than CLIP, might need to reduce batch size

print("Generating embeddings for vocabulary...")

# We don't need autocast here necessarily if the model handles types internally,
# but keeping it safe for mixed precision if configured.
for i in tqdm(range(0, len(ds), BATCH_SIZE)):
    # Slice batch
    batch_data = ds.select(range(i, min(i + BATCH_SIZE, len(ds))))
    batch_ids = [str(j) for j in range(i, i + len(batch_data))]

    # Prepare Metadata and Document Text
    # Chroma requires 'documents' to be strings
    batch_doc_texts = [f"{row['term_label']} : {row['term_note'] or ''}" for row in batch_data]
    batch_metadata = [{"term_label": row['term_label']} for row in batch_data]

    # Prepare Qwen Inputs (List of dicts)
    qwen_inputs = [{"text": text} for text in batch_doc_texts]

    # Generate Embeddings
    # model.process returns a tensor on the correct device
    embeddings = model.process(qwen_inputs)

    # Normalize embeddings (Critical for Cosine Similarity in Chroma)
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

    # Add to Chroma
    collection.add(
        ids=batch_ids,
        embeddings=embeddings.cpu().float().tolist(), # Convert to float32 list
        documents=batch_doc_texts,
        metadatas=batch_metadata
    )

print("finished")

# Choose How Many Keywords to Find

In [ ]:
TERM_COUNT = 10

#Query and Rerank Keywords from Vector Database

In [ ]:
# note: flash attention was installed as a prebuilt wheel in non colab build

import chromadb
import torch
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import os
import gc

# Import the hf Qwen wrapper
from scripts.qwen3_vl_embedding import Qwen3VLEmbedder
from scripts.qwen3_vl_reranker import Qwen3VLReranker


# Load ChromaDB (pre vocab embedded)
chroma_client = chromadb.PersistentClient(path=VDB_PATH)
collection = chroma_client.get_collection(name="aat_terms")

# Load embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Qwen embedding model on {device}...")

embedding_model_name = "Qwen/Qwen3-VL-Embedding-2B"


embedding_model = Qwen3VLEmbedder(
    model_name_or_path=embedding_model_name,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    attn_implementation="flash_attention_2"
)


# load mcam data from huggingface
mcam_data = load_dataset(path="KeeganC/mcam-object-artist-records", split="train")


# Process image query
print("\nProcessing image query...")
art_piece = mcam_data[122] # select art piece to process (index -1)

art_query = {"image": art_piece["Image"], "text":art_piece["Title"], "text":art_piece["Media & Support"]}

query_input = [art_query]

# Generate Image Embedding
image_features = embedding_model.process(query_input)

# Normalize Image Embedding (for chromadb comparison)
image_features = torch.nn.functional.normalize(image_features, p=2, dim=1)

# Query ChromaDB for top n results
results = collection.query(
    query_embeddings=image_features.cpu().float().tolist(),
    n_results=TERM_COUNT
)

# list of the labels and the docs
labels = []
input_docs = []
print("\n--- Top Matches ---")
if results['documents']:
    for doc, metadatas, distance in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        # Chroma "cosine" distance = 1 - similarity
        similarity = (1 - distance) * 100
        print(f"{metadatas['term_label']:<24} {similarity:.2f}% match")
        input_docs.append({"text":doc})
        labels.append(metadatas['term_label'])
else:
    print("No results found.")


print()
print("Unloading embedding model...")
del embedding_model
gc.collect()
torch.cuda.empty_cache()
print("VRAM cleared. Ready to load reranker.")



# Load reranking model
print()
print(f"Loading Qwen reranking model on {device}...")
reranking_model_name = "Qwen/Qwen3-VL-Reranker-2B"

reranking_model = Qwen3VLReranker(model_name_or_path=reranking_model_name,
                                  dtype=torch.bfloat16,
                                  attn_implementation="flash_attention_2")

# Combine queries and documents into the input
inputs = {
    "instruction": "Retrieve Art & Architecture Thesaurus terms relevant to the given image.",
    "query": art_query,
    "documents": input_docs,
    "fps": 1.0
}

print("Processing reranking")
# Generate scores for each word
scores = reranking_model.process(inputs)
print("Reranking Completed")

# sort and print the reranked terms
print("\n--- Top Matches Rerank Scores ---")
sorted_results = sorted(zip(scores, labels), reverse=True)


for score, label in sorted_results:
    print(f"{label:<24} {(score*100):.2f}% match")


Loading Qwen embedding model on cuda...


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]


Processing image query...

--- Top Matches ---
oil paintings            63.13% match
oil by technique         60.06% match
oil painters             54.06% match
oil sketches             53.94% match
oil painting             52.24% match
oil by composition or origin 51.49% match
oil pastels              51.15% match
still lifes              50.58% match
paintings by material or technique 50.34% match
lavender oil             49.98% match

Unloading embedding model...
VRAM cleared. Ready to load reranker.

Loading Qwen reranking model on cuda...


You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Processing reranking
Reranking Completed

--- Top Matches Rerank Scores ---
still lifes              67.97% match
oil painting             65.23% match
oil paintings            58.98% match
lavender oil             55.08% match
paintings by material or technique 53.91% match
oil painters             53.91% match
oil by composition or origin 51.56% match
oil by technique         50.78% match
oil sketches             43.16% match
oil pastels              33.98% match
